# 🐝 Lab 5.2 — Teaching Models to Reason with RL (GRPO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab14_grpo_reasoning.ipynb)

⏱ **Time:** ~2 hours  |  💻 **Runs on:** CPU (Part 7 is ⚡ GPU-optional)

### 🎯 Learning objectives
By the end of this lab you will be able to:
- Explain the 2025–26 **reasoning-model recipe**: SFT warm start → **RL with verifiable rewards (RLVR)** → test-time compute.
- Implement **GRPO** from scratch: group sampling, rule-based rewards, group-relative advantages, the clipped surrogate, and a KL penalty to a frozen reference — *no value network*.
- Diagnose RL training with reward curves, KL curves and the fraction of "zero-signal" groups.
- Contrast on-policy RL with **DPO**, and implement the DPO loss.
- Measure **test-time compute** scaling: pass@N with a verifier and **majority vote (self-consistency)** as a function of N.

### 🧰 Prerequisites
PyTorch training loops, cross-entropy / log-probabilities, a rough idea of what a transformer language model is. Lab 5.1 (LoRA) is helpful but not required.

### 🌍 Why this matters in 2026
The big jump in LLM capability since late 2024 came less from bigger pretraining runs and more from **post-training with reinforcement learning on checkable tasks** (math answers, unit tests, puzzles). DeepSeek-R1 (2025) showed in the open that a simple critic-free algorithm, **GRPO** (introduced in DeepSeekMath, 2024), plus *verifiable* rewards is enough to make long chains of thought emerge. Spending more compute at inference (sampling many answers and voting, or verifying) is the other half of the story. In this lab Buzz the Backprop Bee 🐝 shrinks that whole pipeline down to a laptop: a tiny transformer that learns two-digit addition, first by imitation, then by trial-and-error with rewards.

## 🗺️ Roadmap
| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 | Setup | 3 |
| 1 | The reasoning-model recipe and the GRPO objective | 15 |
| 2 | A verifiable toy world and its reward functions | 10 |
| 3 | A nano-transformer policy with a short SFT warm start | 10 |
| 4 | GRPO from scratch: advantages, clipped loss + KL, training, and an animation of the answer distribution | 35 |
| 5 | DPO for contrast, and why plain DPO can backfire | 20 |
| 6 | Test-time compute: pass@N and majority vote | 15 |
| 7 | ⚡ Optional GPU: TRL `GRPOTrainer` on Qwen2.5-0.5B | 12 |

## 0 · Setup
Everything in Parts 1–6 uses only PyTorch + numpy + matplotlib (preinstalled on Colab). Part 7 installs `trl` only if a GPU is present.

In [ ]:
import os, sys, math, random, time, json, re
import numpy as np
import torch
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})
print(f"🐝 torch {torch.__version__} | device={DEVICE} | FAST={FAST}")

import copy
import torch.nn as nn
import torch.nn.functional as F
torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(8, os.cpu_count() or 1))))

# ---- budgets (FAST shrinks everything for smoke tests) ----
SFT_STEPS  = 300                      # deliberately short: we want a *mediocre* starting policy (~50%)
RL_STEPS   = 80  if FAST else 300
GROUP_SIZE = 8   if FAST else 16      # G: completions sampled per prompt
PROMPTS_PER_STEP = 32
EVAL_N     = 200 if FAST else 500

## Part 1 · The reasoning recipe in one picture

Open reasoning models such as DeepSeek-R1 and Qwen3 document a recipe like the one below. Closed models (OpenAI's o-series, Claude with extended thinking, …) don't publish full details, but are widely believed to use related ideas:

1. **Pretrain** a base model on a lot of text (it *can sometimes* solve problems).
2. **SFT warm start** on a modest number of demonstrations so it follows the output format.
3. **RL with verifiable rewards (RLVR)**: sample answers, *check them with a program* (does the math answer match? do the unit tests pass?), and push probability mass toward what worked.
4. **Test-time compute**: let the model think longer and/or sample several answers and aggregate.

### From policy gradients to GRPO
For a prompt $q$ and a sampled completion $o \sim \pi_\theta(\cdot\mid q)$ with reward $r(q,o)$, the REINFORCE gradient is

$$\nabla_\theta J = \mathbb{E}\big[(r - b)\,\nabla_\theta \log \pi_\theta(o\mid q)\big],$$

where the **baseline** $b$ reduces variance without adding bias. PPO (Schulman et al., 2017) learns $b$ with a separate **value network** — for LLMs that is a second billion-parameter model to train. **GRPO** (Group Relative Policy Optimization, Shao et al., 2024) instead samples a **group** of $G$ completions for the *same* prompt and uses the group itself as the baseline:

$$\hat A_i = \frac{r_i - \operatorname{mean}(r_1,\dots,r_G)}{\operatorname{std}(r_1,\dots,r_G) + \varepsilon}.$$

Every token of completion $i$ gets the same advantage $\hat A_i$. The loss keeps PPO's **clipped importance ratio** $\rho_{i,t} = \pi_\theta(o_{i,t}\mid q, o_{i,<t}) / \pi_{\theta_\text{old}}(o_{i,t}\mid\cdot)$ so we can take a few gradient steps per batch of samples safely, and adds a **KL penalty** to a frozen reference model $\pi_\text{ref}$ (the SFT model):

$$\mathcal{J}_\text{GRPO}(\theta) = \mathbb{E}\Bigg[\frac{1}{G}\sum_{i=1}^{G}\frac{1}{|o_i|}\sum_{t=1}^{|o_i|}\Big(\min\big(\rho_{i,t}\hat A_i,\ \operatorname{clip}(\rho_{i,t},1-\epsilon,1+\epsilon)\hat A_i\big) - \beta\, \mathbb{D}_\text{KL}\big[\pi_\theta \,\|\, \pi_\text{ref}\big]_{t}\Big)\Bigg]$$

with the per-token, always-non-negative "k3" KL estimator $\;\mathbb{D}_{t} = \frac{\pi_\text{ref}}{\pi_\theta} - \log\frac{\pi_\text{ref}}{\pi_\theta} - 1$.

**Why it works so well for LLMs:** no critic to train, the baseline is *prompt-specific* (an easy prompt where every sample succeeds gives zero advantage — no wasted push), and rewards can be simple programs, which are hard to "reward hack" compared with a learned reward model.

In [ ]:
# A picture of group-relative advantages: one prompt, G=8 samples, binary-ish rewards
r = np.array([1.1, 0.1, 0.1, 1.1, 0.0, 0.1, 1.1, 0.1])
A = (r - r.mean()) / (r.std(ddof=1) + 1e-4)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar(range(8), r, color=[HIVE[3] if x > 1 else HIVE[2] for x in r]); ax[0].set_title("rewards in one group"); ax[0].set_xlabel("sample i")
ax[1].bar(range(8), A, color=[HIVE[3] if a > 0 else HIVE[2] for a in A]); ax[1].axhline(0, color=HIVE[1], lw=0.8)
ax[1].set_title("group-relative advantage  Â_i"); ax[1].set_xlabel("sample i")
plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** GRPO is policy gradient with a **group baseline**: sample $G$ answers per prompt, normalize their rewards within the group, and push up the better-than-average ones. No critic network needed.

## Part 2 · A verifiable toy world: two-digit addition 🧮

Our "environment" is a prompt like `47+38=` and the model must emit the answer followed by an end token `;` (e.g. `85;`). The vocabulary is just 14 characters.

Two **rule-based rewards**, exactly in the spirit of DeepSeek-R1 (which used an *accuracy* reward plus a *format* reward):

| reward | value | checks |
|---|---|---|
| format | 0.1 | completion matches the regex `\d{1,3};` (digits, then the end token, nothing else) |
| correctness | 1.0 | well-formed **and** the number equals $a+b$ |

So a completion scores $r \in \{0, 0.1, 1.1\}$.

In [ ]:
VOCAB = list("0123456789+=;_")        # '_' is padding, ';' ends an answer
stoi = {c: i for i, c in enumerate(VOCAB)}
PAD, EOS = stoi["_"], stoi[";"]
MAX_NEW = 4                           # at most 3 digits + ';'

def make_problem(rng):
    a, b = rng.randint(0, 99), rng.randint(0, 99)
    return f"{a}+{b}=", str(a + b)

def encode(s):  return [stoi[c] for c in s]

def decode_completion(ids):
    """Turn generated ids into a string, stopping at EOS or padding."""
    s = ""
    for t in ids.tolist():
        if t == PAD: break
        s += VOCAB[t]
        if t == EOS: break
    return s

rng = random.Random(0)
print([make_problem(rng) for _ in range(5)])

### 🧪 Your Turn #1 — write the reward functions
Implement `format_reward(completion)` and `correctness_reward(completion, answer)`.
*Hints:* `re.fullmatch(r"\d{1,3};", s)`; correctness should be `0.0` if the format is wrong (so the model can't get credit for `85;;;` or `85` without the end token).

In [ ]:
def format_reward(completion: str) -> float:
    # TODO: return 0.1 if completion is 1-3 digits followed by ';' (and nothing else), else 0.0
    raise NotImplementedError

def correctness_reward(completion: str, answer: str) -> float:
    # TODO: return 1.0 if well-formed AND the digits equal `answer`, else 0.0
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def format_reward(completion: str) -> float:
    return 0.1 if re.fullmatch(r"\d{1,3};", completion) else 0.0

def correctness_reward(completion: str, answer: str) -> float:
    if not format_reward(completion):
        return 0.0
    return 1.0 if completion[:-1] == answer else 0.0

def total_reward(completion, answer):
    return format_reward(completion) + correctness_reward(completion, answer)

In [ ]:
# ✔️ Checkpoint
assert format_reward("85;") == 0.1 and format_reward("85") == 0.0 and format_reward("8a;") == 0.0
assert format_reward("85;;") == 0.0 and format_reward("1234;") == 0.0
assert correctness_reward("85;", "85") == 1.0 and correctness_reward("84;", "85") == 0.0
assert correctness_reward("85", "85") == 0.0, "no credit without the end token"
assert abs(total_reward("85;", "85") - 1.1) < 1e-9
print("✔️ rewards look right — Buzz approves 🐝")

> **🍯 Key takeaway:** Verifiable rewards are just programs. Make them strict (no credit for a malformed answer) or the policy will find the loophole.

## Part 3 · A tiny policy and a *short* SFT warm start

Our policy is a 2-layer, 128-dim causal transformer (~0.4M parameters) — a "nano-LLM". We give it a short supervised warm-up on correct examples (cross-entropy on answer tokens only, prompt tokens masked out — the same loss masking you used in Lab 5.1).

We **stop SFT early on purpose**: the point of RLVR is to improve a model that *sometimes* succeeds. If the policy never produces a correct answer, every group has identical rewards, all advantages are zero, and RL has nothing to learn from. (This is why R1-style RL works far better on strong base models.)

In [ ]:
class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x, mask):
        h = self.ln1(x)
        x = x + self.attn(h, h, h, attn_mask=mask, need_weights=False)[0]
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self, V=len(VOCAB), d=128, L=2, h=4, T=16):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(T, d)
        self.blocks = nn.ModuleList([Block(d, h) for _ in range(L)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V)
    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=idx.device), 1)  # causal
        for b in self.blocks:
            x = b(x, mask)
        return self.head(self.ln(x))

seed_everything(0)
policy = TinyGPT().to(DEVICE)
print(f"policy parameters: {sum(p.numel() for p in policy.parameters()):,}")

In [ ]:
def sft_batch(rng, B):
    seqs, masks = [], []
    for _ in range(B):
        p, a = make_problem(rng)
        ids = encode(p + a + ";")
        seqs.append(ids); masks.append([0] * len(p) + [1] * (len(a) + 1))   # loss only on the answer
    T = max(map(len, seqs))
    x = torch.full((B, T), PAD); m = torch.zeros(B, T)
    for i, (s, mm) in enumerate(zip(seqs, masks)):
        x[i, :len(s)] = torch.tensor(s); m[i, :len(mm)] = torch.tensor(mm, dtype=torch.float)
    return x.to(DEVICE), m.to(DEVICE)

@torch.no_grad()
def generate(model, prompts, max_new=MAX_NEW, temperature=1.0, greedy=False):
    """Sample completions. Prompts of equal length are batched together (no padding needed)."""
    out = [None] * len(prompts)
    by_len = {}
    for i, p in enumerate(prompts):
        by_len.setdefault(len(p), []).append(i)
    for _, idxs in by_len.items():
        x = torch.tensor([encode(prompts[i]) for i in idxs], device=DEVICE)
        done = torch.zeros(len(idxs), dtype=torch.bool, device=DEVICE)
        gen = []
        for _ in range(max_new):
            logits = model(x)[:, -1]
            nxt = logits.argmax(-1) if greedy else torch.multinomial(F.softmax(logits / temperature, -1), 1).squeeze(-1)
            nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)   # after EOS, emit padding
            gen.append(nxt); done |= nxt == EOS
            x = torch.cat([x, nxt[:, None]], 1)
        gen = torch.stack(gen, 1)
        for j, i in enumerate(idxs):
            out[i] = gen[j]
    return torch.stack(out)

def evaluate(model, n=EVAL_N, seed=123):
    """Greedy accuracy on a fixed held-out set of problems."""
    model.eval()
    rng = random.Random(seed); probs = [make_problem(rng) for _ in range(n)]
    comps = generate(model, [p for p, _ in probs], greedy=True)
    return float(np.mean([correctness_reward(decode_completion(c), a) for c, (_, a) in zip(comps, probs)]))

opt = torch.optim.AdamW(policy.parameters(), lr=1e-3)
rng = random.Random(1); sft_losses = []
t0 = time.time()
for step in range(SFT_STEPS):
    x, m = sft_batch(rng, 128)
    logits = policy(x[:, :-1])
    ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1), reduction="none")
    loss = (ce * m[:, 1:].reshape(-1)).sum() / m[:, 1:].sum()
    opt.zero_grad(); loss.backward(); opt.step()
    sft_losses.append(loss.item())
sft_acc = evaluate(policy)
print(f"SFT done in {time.time()-t0:.1f}s | final loss {sft_losses[-1]:.3f} | greedy accuracy {sft_acc:.1%}")

plt.figure(figsize=(6, 3)); plt.plot(sft_losses); plt.yscale("log")
plt.title("SFT warm start (answer-token cross-entropy)"); plt.xlabel("step"); plt.show()

In [ ]:
# Peek at what the warm-started policy does (sampled at temperature 1)
rng = random.Random(5)
for _ in range(6):
    p, a = make_problem(rng)
    samples = [decode_completion(c) for c in generate(policy, [p] * 6)]
    print(f"{p:7s} truth={a:>3s}  samples={samples}")

> **🍯 Key takeaway:** RL can only amplify behaviour the model already produces *sometimes*: start from a policy with non-zero success, or every advantage is zero.

## Part 4 · GRPO from scratch 🏗️

One GRPO step:
1. Draw $B$ prompts; sample $G$ completions for each from the current policy (temperature 1 — exploration matters!).
2. Score every completion with the reward functions.
3. Compute **group-relative advantages** (normalize within each prompt's group).
4. Record $\log\pi_{\theta_\text{old}}$ (the sampler) and $\log\pi_\text{ref}$ (frozen SFT copy) for every generated token.
5. Take $\mu$ gradient steps on the clipped surrogate minus $\beta\cdot$KL, averaging over tokens of each completion, then over completions.

### 🧪 Your Turn #2 — group-relative advantages
`rewards` has shape `(B, G)`. Return advantages of the same shape: subtract each row's mean and divide by each row's (unbiased) std plus `eps`.

In [ ]:
def group_advantages(rewards: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    # TODO: normalize within each group (row). Hint: .mean(dim=1, keepdim=True), .std(dim=1, keepdim=True)
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def group_advantages(rewards: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True)
    return (rewards - mean) / (std + eps)

In [ ]:
# ✔️ Checkpoint
R = torch.tensor([[1.1, 0.1, 0.1, 1.1], [0.1, 0.1, 0.1, 0.1]])
A = group_advantages(R)
assert A.shape == R.shape
assert torch.allclose(A.mean(1), torch.zeros(2), atol=1e-6), "each group should be centered"
assert torch.allclose(A[1], torch.zeros(4)), "identical rewards -> zero advantage (no learning signal!)"
assert A[0, 0] > 0 and A[0, 1] < 0
print("✔️ advantages OK:", A[0].numpy().round(3))

### 🧪 Your Turn #3 — the clipped surrogate + KL loss
Given per-token log-probs under the current policy `logp`, the sampler `old_logp`, the reference `ref_logp`, a `mask` of real (non-padding) tokens — all shape `(N, T)` — and per-sequence advantages `adv` of shape `(N,)`, return a tuple `(loss, kl)`: the scalar **loss** (negative objective) and the detached per-token KL tensor (we log it):

$$\text{loss} = -\frac{1}{N}\sum_i \frac{\sum_t m_{i,t}\,\big[\min(\rho\hat A_i, \text{clip}(\rho)\hat A_i) - \beta\,(e^{\,\text{ref}-\text{logp}} - (\text{ref}-\text{logp}) - 1)\big]}{\sum_t m_{i,t}}$$

In [ ]:
def grpo_loss(logp, old_logp, ref_logp, mask, adv, clip_eps=0.2, beta=0.02):
    # TODO 1: ratio = exp(logp - old_logp)
    # TODO 2: surrogate = min(ratio * A, clamp(ratio, 1-eps, 1+eps) * A)   (A broadcast over tokens)
    # TODO 3: k3 KL per token: exp(ref - logp) - (ref - logp) - 1
    # TODO 4: masked mean over tokens of each sequence, then mean over sequences
    # TODO 5: return (loss, kl.detach())  where loss = -(objective) is a scalar and kl is the (N, T) per-token KL
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def grpo_loss(logp, old_logp, ref_logp, mask, adv, clip_eps=0.2, beta=0.02):
    ratio = torch.exp(logp - old_logp)
    A = adv[:, None]
    surrogate = torch.min(ratio * A, torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * A)
    log_r = ref_logp - logp
    kl = torch.exp(log_r) - log_r - 1
    per_token = surrogate - beta * kl
    per_seq = (per_token * mask).sum(1) / mask.sum(1).clamp(min=1)
    return -per_seq.mean(), kl.detach()

In [ ]:
# ✔️ Checkpoint — compare with a slow, loop-based reference implementation
def _ref_loss(logp, old, ref, mask, adv, eps=0.2, beta=0.02):
    tot = 0.0
    for i in range(logp.shape[0]):
        num, den = 0.0, 0.0
        for t in range(logp.shape[1]):
            if mask[i, t] == 0: continue
            r = math.exp(logp[i, t] - old[i, t]); a = adv[i].item()
            s = min(r * a, min(max(r, 1 - eps), 1 + eps) * a)
            lr = (ref[i, t] - logp[i, t]).item(); k = math.exp(lr) - lr - 1
            num += s - beta * k; den += 1
        tot += num / max(den, 1)
    return -tot / logp.shape[0]

g = torch.Generator().manual_seed(0)
lp, old, rf = (-torch.rand(5, 4, generator=g) * 3 for _ in range(3))
msk = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0], [1, 1, 1, 1], [1, 0, 0, 0], [1, 1, 1, 1]], dtype=torch.float)
adv = torch.randn(5, generator=g)
ours, _ = grpo_loss(lp, old, rf, msk, adv)
assert abs(ours.item() - _ref_loss(lp, old, rf, msk, adv)) < 1e-5
# when policy == sampler == reference, the loss is just -mean(advantage) and KL is 0
same, kl0 = grpo_loss(lp, lp, lp, msk, adv)
assert abs(same.item() + adv.mean().item()) < 1e-5 and kl0.abs().max() < 1e-6
print("✔️ GRPO loss matches the reference implementation")

Now the per-token log-probabilities of a batch of completions, and the training loop itself.

In [ ]:
def completion_logprobs(model, prompts, comps):
    """Per-token log pi(o_t | q, o_<t) for generated tokens. Returns (logp, mask), both (N, MAX_NEW)."""
    N, T = comps.shape
    logp = torch.zeros(N, T, device=DEVICE)
    by_len = {}
    for i, p in enumerate(prompts):
        by_len.setdefault(len(p), []).append(i)
    for ln, idxs in by_len.items():
        idx_t = torch.tensor(idxs, device=DEVICE)
        x = torch.cat([torch.tensor([encode(prompts[i]) for i in idxs], device=DEVICE), comps[idx_t]], 1)
        logits = model(x[:, :-1])[:, ln - 1:]          # positions that predict the completion tokens
        lp = torch.log_softmax(logits, -1).gather(-1, comps[idx_t][..., None]).squeeze(-1)
        logp = logp.index_copy(0, idx_t, lp)
    # mask = real tokens up to and including EOS
    is_pad = comps == PAD
    mask = (~is_pad).float()
    return logp, mask

# Freeze a copy of the SFT policy as the reference (and keep it for later comparisons)
sft_policy = copy.deepcopy(policy).eval()
for p in sft_policy.parameters():
    p.requires_grad_(False)
ref_model = sft_policy

### 🔮 Predict-then-run
Before running the training loop, guess:
1. Will the **mean sampled reward** go up smoothly or noisily?
2. What happens to the fraction of groups where *all G samples get the same reward* (zero advantage → zero gradient) as the policy improves?
3. Will the KL to the reference grow, stay flat, or shrink?

In [ ]:
seed_everything(7)
opt = torch.optim.AdamW(policy.parameters(), lr=3e-4)
CLIP_EPS, BETA, MU = 0.2, 0.02, 2             # MU = gradient steps per batch of samples
rng = random.Random(7)
hist = {"reward": [], "acc_sampled": [], "kl": [], "zero_groups": [], "eval": [], "eval_step": []}
err_snapshots = []          # (step, array of sampled-answer errors) for the animation below

t0 = time.time()
for step in range(RL_STEPS):
    # 1) sample a group of G completions for each of B prompts
    probs = [make_problem(rng) for _ in range(PROMPTS_PER_STEP)]
    prompts = [p for p, _ in probs for _ in range(GROUP_SIZE)]
    answers = [a for _, a in probs for _ in range(GROUP_SIZE)]
    policy.eval()
    comps = generate(policy, prompts)
    texts = [decode_completion(c) for c in comps]

    # 2) verifiable rewards  3) group-relative advantages
    rewards = torch.tensor([total_reward(t, a) for t, a in zip(texts, answers)], device=DEVICE)
    R = rewards.view(PROMPTS_PER_STEP, GROUP_SIZE)
    adv = group_advantages(R).view(-1)

    # 4) log-probs of the sampler (old policy) and the frozen reference
    with torch.no_grad():
        old_logp, mask = completion_logprobs(policy, prompts, comps)
        ref_logp, _ = completion_logprobs(ref_model, prompts, comps)

    # 5) a few clipped policy-gradient steps on this batch
    policy.train()
    for _ in range(MU):
        logp, _ = completion_logprobs(policy, prompts, comps)
        loss, kl = grpo_loss(logp, old_logp, ref_logp, mask, adv, CLIP_EPS, BETA)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step()

    if step % max(1, RL_STEPS // 30) == 0 or step == RL_STEPS - 1:
        errs = [int(t[:-1]) - int(a) if format_reward(t) else 99 for t, a in zip(texts, answers)]
        err_snapshots.append((step, np.array(errs)))
    hist["reward"].append(rewards.mean().item())
    hist["acc_sampled"].append((rewards > 1).float().mean().item())
    hist["kl"].append(((kl * mask).sum() / mask.sum()).item())
    hist["zero_groups"].append((R.std(1) == 0).float().mean().item())
    if step % 25 == 0 or step == RL_STEPS - 1:
        hist["eval"].append(evaluate(policy)); hist["eval_step"].append(step)
        print(f"step {step:4d} | mean reward {hist['reward'][-1]:.3f} | greedy acc {hist['eval'][-1]:.1%} "
              f"| KL {hist['kl'][-1]:.4f} | {time.time()-t0:.0f}s")

grpo_acc = evaluate(policy)
print(f"\n🐝 greedy accuracy: SFT {sft_acc:.1%}  ->  SFT+GRPO {grpo_acc:.1%}")

In [ ]:
def smooth(x, k=10):
    x = np.asarray(x); k = max(1, min(k, len(x)))
    return np.convolve(x, np.ones(k) / k, mode="valid")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(hist["reward"], alpha=0.3, color=HIVE[0]); ax[0].plot(smooth(hist["reward"]), color=HIVE[0], lw=2, label="mean reward (sampled)")
ax[0].plot(hist["eval_step"], hist["eval"], "o-", color=HIVE[4], label="greedy accuracy (held-out)")
ax[0].set_title("Reward goes up"); ax[0].set_xlabel("GRPO step"); ax[0].legend()
ax[1].plot(smooth(hist["zero_groups"]), color=HIVE[2]); ax[1].set_title("fraction of zero-signal groups"); ax[1].set_xlabel("GRPO step")
ax[2].plot(smooth(hist["kl"]), color=HIVE[3]); ax[2].set_title("KL(π_θ ‖ π_ref) per token"); ax[2].set_xlabel("GRPO step")
plt.tight_layout(); plt.show()

In [ ]:
# ✔️ Checkpoint — RL should measurably improve the policy
first, last = np.mean(hist["reward"][:10]), np.mean(hist["reward"][-10:])
print(f"mean sampled reward: first 10 steps {first:.3f} -> last 10 steps {last:.3f}")
assert last > first + (0.02 if FAST else 0.1), "reward should rise during GRPO"

### 🎬 Watch the answer distribution sharpen
Each frame is a histogram of **(sampled answer − true answer)** over the whole batch at one point in training (malformed answers are lumped into the "bad" bar on the right). At the start the mass is smeared over off-by-one and off-by-ten mistakes; as GRPO proceeds, it piles up on **0**.

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display

BINS = list(range(-12, 13))
def err_hist(errs):
    clipped = np.clip(errs[errs != 99], -12, 12)
    counts = np.array([(clipped == b).sum() for b in BINS] + [(errs == 99).sum()]) / len(errs)
    return counts

fig, ax = plt.subplots(figsize=(9, 3.6))
xs = BINS + [14]
bars = ax.bar(xs, err_hist(err_snapshots[0][1]), color=[HIVE[3] if b == 0 else HIVE[0] for b in BINS] + [HIVE[2]])
ax.set_xticks([-12, -10, -5, -1, 0, 1, 5, 10, 14], ["≤-12", "-10", "-5", "-1", "0", "+1", "+5", "+10", "bad"])
ax.set_ylim(0, 1); ax.set_xlabel("sampled answer − true answer"); ax.set_ylabel("fraction of samples")
def update(i):
    step, errs = err_snapshots[i]
    for b, h in zip(bars, err_hist(errs)):
        b.set_height(h)
    ax.set_title(f"GRPO step {step}:  {np.mean(errs == 0):.0%} of sampled answers exactly right")
    return bars
anim = animation.FuncAnimation(fig, update, frames=len(err_snapshots), interval=250, blit=False)
display(HTML(anim.to_jshtml()))
plt.close(fig)

**What just happened?**
- The sampled reward climbs as probability mass moves to correct answers. Greedy accuracy rises too, though more noisily.
- The zero-signal fraction rises: as the policy gets good, more groups are "all correct" and contribute nothing. That's the motivation for **dynamic sampling** in DAPO (Yu et al., 2025), which filters out such groups and resamples, and for curricula that keep problems at the edge of the model's ability.
- KL grows slowly from 0: the policy drifts away from the SFT reference, and $\beta$ controls how far. (Several 2025 recipes set $\beta = 0$ for verifiable tasks; the KL term matters more when rewards are hackable.)

> 🐝 **Honesty note.** On a toy task this easy, *more SFT* would also get you there. RL shines when you **can check answers but can't write demonstrations** (new math problems, failing unit tests, puzzles) and when the model's own successful samples are better training data than human ones.

#### Known subtleties (2025 research)
- **Length bias:** dividing by $|o_i|$ makes long *wrong* answers less penalized per token. "Dr. GRPO" (Liu et al., 2025) removes the per-sequence length normalization and the std division.
- **Std normalization** up-weights prompts with tiny reward variance (very easy/very hard), which may or may not be what you want.

> **🍯 Key takeaway:** Reward goes up and the zero-signal fraction rises as the model masters prompts. The KL term is the leash that keeps the policy near its SFT starting point.

## Part 5 · Contrast: DPO — preference learning without RL

**Direct Preference Optimization** (Rafailov et al., 2023) starts from the KL-regularized RL objective, notes that its optimal policy satisfies $r(x,y) = \beta \log\frac{\pi^*(y\mid x)}{\pi_\text{ref}(y\mid x)} + \beta\log Z(x)$, and plugs this into the Bradley–Terry preference model. The partition function cancels, leaving a simple classification loss on **preference pairs** $(x, y_w \succ y_l)$:

$$\mathcal{L}_\text{DPO} = -\log\sigma\Big(\beta\Big[\big(\log\pi_\theta(y_w|x) - \log\pi_\text{ref}(y_w|x)\big) - \big(\log\pi_\theta(y_l|x) - \log\pi_\text{ref}(y_l|x)\big)\Big]\Big)$$

| | GRPO / PPO (online RL) | DPO (offline) |
|---|---|---|
| data | fresh samples from the current policy | fixed dataset of chosen/rejected pairs |
| signal | scalar reward (program or reward model) | pairwise preference |
| extra models | reference (+ value net for PPO) | reference only |
| typical use 2026 | reasoning, code, agents (verifiable) | chat style, helpfulness, safety tuning |

### 🧪 Your Turn #4 — implement the DPO loss
Inputs are **sequence** log-probs (summed over completion tokens), shape `(N,)`. Return the mean loss and the "implicit reward margin" $\beta[(\ldots)_w - (\ldots)_l]$. Hint: `F.logsigmoid`.

In [ ]:
def dpo_loss(pol_chosen, pol_rejected, ref_chosen, ref_rejected, beta=0.1):
    # TODO: margin = beta * ((pol_chosen - ref_chosen) - (pol_rejected - ref_rejected))
    # TODO: loss = -logsigmoid(margin).mean()
    # TODO: return (loss, margin.detach())
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def dpo_loss(pol_chosen, pol_rejected, ref_chosen, ref_rejected, beta=0.1):
    margin = beta * ((pol_chosen - ref_chosen) - (pol_rejected - ref_rejected))
    return -F.logsigmoid(margin).mean(), margin.detach()

In [ ]:
# ✔️ Checkpoint
z = torch.zeros(3)
l0, m0 = dpo_loss(z, z, z, z)
assert abs(l0.item() - math.log(2)) < 1e-6, "policy == reference -> loss = log 2"
l1, m1 = dpo_loss(torch.tensor([-1.0]), torch.tensor([-5.0]), torch.tensor([-2.0]), torch.tensor([-2.0]), beta=0.5)
assert abs(m1.item() - 2.0) < 1e-6 and abs(l1.item() - (-F.logsigmoid(torch.tensor(2.0))).item()) < 1e-6  # 0.5*((−1+2) − (−5+2)) = 2
print("✔️ DPO loss OK")

**Mini-demo:** build preference pairs *from the SFT model's own samples* (a correct sample is "chosen", an incorrect one "rejected" — this is how many "RL-free" reasoning pipelines bootstrap) and train two fresh copies of the SFT policy:

1. **plain DPO**, and
2. **DPO + NLL**: add a small negative-log-likelihood (SFT) term on the *chosen* answers, as in Iterative RPO (Pang et al., 2024).

### 🔮 Predict-then-run
The DPO loss only cares about the *margin* between chosen and rejected. Can the loss go down while the log-probability of the **chosen** answer *also* goes down? What would that do to accuracy?

In [ ]:
def make_pairs(model, n_prompts, G=8, seed=11):
    rng = random.Random(seed); pairs = []
    for _ in range(n_prompts):
        p, a = make_problem(rng)
        comps = generate(model, [p] * G)
        good = [c for c in comps if correctness_reward(decode_completion(c), a) > 0]
        bad = [c for c in comps if correctness_reward(decode_completion(c), a) == 0]
        if good and bad:
            pairs.append((p, good[0], bad[0]))
    return pairs

def seq_logp(model, prompts, comps):
    lp, m = completion_logprobs(model, prompts, comps)
    return (lp * m).sum(1), m.sum(1)

seed_everything(11)
pairs = make_pairs(sft_policy, 300 if FAST else 600)
print(f"{len(pairs)} preference pairs; e.g. {pairs[0][0]} chosen={decode_completion(pairs[0][1])!r} rejected={decode_completion(pairs[0][2])!r}")

def train_dpo(nll_weight, lr=1e-4, epochs=2, beta=0.1, seed=0):
    seed_everything(seed); data = list(pairs)
    model = copy.deepcopy(sft_policy).train()
    for p_ in model.parameters(): p_.requires_grad_(True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = {"loss": [], "pref_acc": [], "chosen_logp_change": []}
    for _ in range(epochs):
        random.shuffle(data)
        for i in range(0, len(data), 32):
            chunk = data[i:i + 32]
            P = [c[0] for c in chunk]; W = torch.stack([c[1] for c in chunk]); Lr = torch.stack([c[2] for c in chunk])
            with torch.no_grad():
                (rw, _), (rl, _) = seq_logp(sft_policy, P, W), seq_logp(sft_policy, P, Lr)
            (pw, nw), (pl, _) = seq_logp(model, P, W), seq_logp(model, P, Lr)
            loss, margin = dpo_loss(pw, pl, rw, rl, beta=beta)
            nll = -(pw / nw).mean()                      # per-token NLL of the chosen answers
            opt.zero_grad(); (loss + nll_weight * nll).backward(); opt.step()
            hist["loss"].append(loss.item()); hist["pref_acc"].append((margin > 0).float().mean().item())
            hist["chosen_logp_change"].append((pw - rw).mean().item())
    return model, hist

dpo_policy, dpo_hist = train_dpo(nll_weight=0.0)
rpo_policy, rpo_hist = train_dpo(nll_weight=1.0)
dpo_acc, rpo_acc = evaluate(dpo_policy), evaluate(rpo_policy)
print(f"greedy accuracy  SFT {sft_acc:.1%} | DPO {dpo_acc:.1%} | DPO+NLL {rpo_acc:.1%} | GRPO {grpo_acc:.1%}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.3))
for h, name, col in [(dpo_hist, "DPO", HIVE[2]), (rpo_hist, "DPO+NLL", HIVE[3])]:
    ax[0].plot(h["loss"], color=col, label=name); ax[1].plot(h["pref_acc"], color=col, label=name)
    ax[2].plot(h["chosen_logp_change"], color=col, label=name)
ax[0].set_title("DPO loss"); ax[1].set_title("preference accuracy (margin > 0)")
ax[2].axhline(0, color=HIVE[1], lw=0.8); ax[2].set_title("log π(chosen) − log π_ref(chosen)")
for a in ax: a.set_xlabel("step"); a.legend()
plt.tight_layout(); plt.show()

**What you (probably) saw:** both runs separate chosen from rejected (preference accuracy ↑), but plain DPO does it largely by pushing *both* likelihoods down — the chosen log-prob drifts below the reference and probability leaks to wrong answers, so accuracy stalls or even *drops* (re-run `train_dpo(0.0, lr=3e-4, epochs=3)` to watch it collapse). This "likelihood displacement" is a well-documented DPO failure mode; anchoring with an NLL term on the chosen responses (or a lower learning rate / larger β) keeps the good answers likely. On-policy RL avoids it by construction because it keeps sampling from the current policy.

> **🍯 Key takeaway:** DPO is offline and simple, but it optimizes a *margin*, so it can push down the chosen answers too. Anchoring with an NLL term fixes that here; online RL avoids it by sampling from the current policy.

## Part 6 · Test-time compute: best-of-N and majority vote 🗳️

Instead of (or in addition to) training, we can spend compute at inference:

- **pass@N / best-of-N with a verifier:** sample $N$ answers; succeed if *any* is correct. Requires a checker (here: the reward function). The unbiased estimator from $n \ge N$ samples with $c$ correct is $\text{pass@}N = 1 - \binom{n-c}{N}\big/\binom{n}{N}$ (Chen et al., 2021).
- **Majority vote / self-consistency** (Wang et al., 2023): no verifier needed — sample $N$ answers and return the most common one. Works when errors are *scattered* but the right answer is *consistent*.

### 🧪 Your Turn #5 — majority vote
Return the most common well-formed answer in a list of completions (ignore malformed ones; break ties by first occurrence). Return `None` if nothing is well-formed.

In [ ]:
def majority_vote(completions):
    # TODO: keep completions with format_reward > 0, count them, return the most common (ties -> earliest)
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
from collections import Counter
def majority_vote(completions):
    valid = [c for c in completions if format_reward(c) > 0]
    if not valid:
        return None
    counts = Counter(valid)
    best = max(counts.values())
    return next(c for c in valid if counts[c] == best)

In [ ]:
# ✔️ Checkpoint
assert majority_vote(["12;", "13;", "12;", "x"]) == "12;"
assert majority_vote(["7;", "8;"]) == "7;"
assert majority_vote(["", "1a"]) is None
print("✔️ majority vote OK")

### 🔮 Predict-then-run
For the SFT policy (~50% single-sample accuracy), how will **pass@N** and **majority@N** grow as N goes 1 → 32? Which one saturates higher? Will GRPO's advantage over SFT grow or shrink as N increases?

In [ ]:
def pass_at_k(n, c, k):
    if n - c < k: return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def test_time_curves(model, Ns=(1, 2, 4, 8, 16, 32), n_problems=150 if FAST else 300, seed=99):
    model.eval()
    rng = random.Random(seed); probs = [make_problem(rng) for _ in range(n_problems)]
    nmax = max(Ns)
    prompts = [p for p, _ in probs for _ in range(nmax)]
    comps = [decode_completion(c) for c in generate(model, prompts)]
    passk = {N: [] for N in Ns}; maj = {N: [] for N in Ns}
    for j, (_, a) in enumerate(probs):
        samples = comps[j * nmax:(j + 1) * nmax]
        c = sum(correctness_reward(s, a) > 0 for s in samples)
        for N in Ns:
            passk[N].append(pass_at_k(nmax, c, N))
            maj[N].append(float(majority_vote(samples[:N]) == a + ";"))
    return Ns, [np.mean(passk[N]) for N in Ns], [np.mean(maj[N]) for N in Ns]

Ns, sft_pass, sft_maj = test_time_curves(sft_policy)
_,  rl_pass,  rl_maj  = test_time_curves(policy)
plt.figure(figsize=(7, 4))
plt.plot(Ns, sft_pass, "o--", color=HIVE[0], label="SFT  pass@N (verifier)")
plt.plot(Ns, sft_maj, "o-", color=HIVE[0], label="SFT  majority@N")
plt.plot(Ns, rl_pass, "s--", color=HIVE[4], label="GRPO pass@N (verifier)")
plt.plot(Ns, rl_maj, "s-", color=HIVE[4], label="GRPO majority@N")
plt.xscale("log", base=2); plt.xlabel("N samples per problem"); plt.ylabel("accuracy"); plt.ylim(0, 1.02)
plt.title("Test-time compute scaling"); plt.legend(); plt.show()
for N, a, b, c, d in zip(Ns, sft_pass, sft_maj, rl_pass, rl_maj):
    print(f"N={N:2d}  SFT pass {a:.2f} maj {b:.2f} | GRPO pass {c:.2f} maj {d:.2f}")

**Reading the plot.**
- pass@N rises fastest — a verifier is a superpower. Majority vote helps as long as the right answer is the *plurality* answer.
- RL mostly improves the **N = 1** end: it turns "sometimes right" into "usually right". At large N the curves often move closer together: several 2025 papers (e.g. Yue et al., 2025, *"Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base Model?"*) found that RLVR sharpens the distribution toward answers the base model could already sample, rather than creating entirely new abilities. Whether that holds at frontier scale and with long training is still debated.

> **🍯 Key takeaway:** A verifier is a superpower: pass@N climbs fast, majority vote helps when the right answer is the most common one, and RL mostly lifts the N = 1 end.

## Part 7 · ⚡ GPU recommended — GRPO on a real LLM with TRL

The same algorithm at real scale using Hugging Face **TRL**'s `GRPOTrainer` on `Qwen/Qwen2.5-0.5B-Instruct`. The task is a tiny "countdown"-style arithmetic set with a **format reward** (answer inside `<answer>…</answer>`) and a **correctness reward**. On a free Colab T4 this takes ~15–25 minutes for 100 steps; on CPU the cell **skips automatically**.

> TRL's API moves quickly. If an argument name errors, check the current [GRPOTrainer docs](https://huggingface.co/docs/trl/main/en/grpo_trainer). The core concepts map 1:1 onto Part 4: `num_generations` = $G$, `beta` = KL coefficient, `epsilon` = clip range, reward functions = your verifiers.

In [ ]:
RUN_TRL = torch.cuda.is_available() and not FAST
if not RUN_TRL:
    print("⏭️  No GPU (or FAST mode) — skipping the TRL GRPO run. Everything above ran on CPU.")
else:
    import importlib.util, subprocess
    if importlib.util.find_spec("trl") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trl", "datasets"], check=True)
    from datasets import Dataset
    from trl import GRPOConfig, GRPOTrainer

    def make_countdown(rng):
        nums = [rng.randint(1, 20) for _ in range(3)]
        a, b, c = nums
        ops = rng.choice([("+", "+"), ("+", "-"), ("*", "+"), ("-", "*")])
        target = eval(f"{a}{ops[0]}{b}{ops[1]}{c}")
        q = (f"Compute {a} {ops[0]} {b} {ops[1]} {c} (standard operator precedence). "
             "Think briefly, then give the final integer inside <answer></answer> tags.")
        return {"prompt": [{"role": "user", "content": q}], "target": str(target)}

    rng = random.Random(0)
    train_ds = Dataset.from_list([make_countdown(rng) for _ in range(2000)])

    def format_reward_fn(completions, **kwargs):
        return [0.2 if re.search(r"<answer>\s*-?\d+\s*</answer>", c[0]["content"]) else 0.0 for c in completions]

    def correctness_reward_fn(completions, target, **kwargs):
        out = []
        for c, t in zip(completions, target):
            m = re.search(r"<answer>\s*(-?\d+)\s*</answer>", c[0]["content"])
            out.append(1.0 if m and m.group(1) == t else 0.0)
        return out

    cfg = GRPOConfig(
        output_dir="grpo-qwen-countdown", learning_rate=1e-6, beta=0.04, epsilon=0.2,
        num_generations=8, per_device_train_batch_size=8, gradient_accumulation_steps=2,
        max_completion_length=192, max_steps=100, logging_steps=5, report_to="none",
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    )
    trainer = GRPOTrainer(model="Qwen/Qwen2.5-0.5B-Instruct", reward_funcs=[format_reward_fn, correctness_reward_fn],
                          args=cfg, train_dataset=train_ds)
    trainer.train()
    logs = [l for l in trainer.state.log_history if "reward" in l]
    plt.plot([l["step"] for l in logs], [l["reward"] for l in logs], "o-")
    plt.title("TRL GRPO on Qwen2.5-0.5B: mean reward"); plt.xlabel("step"); plt.show()

> **🍯 Key takeaway:** TRL's `GRPOTrainer` is the same algorithm at scale: `num_generations` is $G$, `beta` is the KL weight, `epsilon` is the clip range, and reward functions are your verifiers.

## 🏆 Boss Level
1. **Dr. GRPO vs GRPO.** Remove the per-sequence length normalization (use a constant denominator, e.g. `MAX_NEW`) and the std division in the advantages. Compare reward curves across 3 seeds. Does it matter on this short-answer task? Why might it matter much more for long chains of thought?
2. **Teach it to "think".** Change the target format to a scratchpad like `47+38=7+8=15,4+3+1=8|85;` (ones digit, then tens with carry, then the answer). SFT on a *small* number of scratchpad examples, then GRPO rewarding only the final answer. Does RL make the scratchpad more reliable? Does accuracy on 3-digit addition improve?
3. **Reward hacking on purpose.** Replace correctness with a sloppy verifier (e.g. "answer has the right number of digits") and watch the policy exploit it. Then add the KL term back with a large $\beta$ — how much does it help?

## 🍯 Key Takeaways
- **RLVR recipe:** SFT warm start → sample → **verify with a program** → reinforce. Cheap, precise rewards are why reasoning RL took off in 2025.
- **GRPO = PPO without the critic:** the group mean is the baseline, $\hat A_i = (r_i-\bar r)/\sigma_r$, with the clipped ratio and a KL leash kept.
- **No variance, no signal:** groups where every sample gets the same reward contribute nothing, so curricula and dynamic sampling keep problems at the edge of ability.
- **Strict verifiers or reward hacking:** give credit only for exactly-right, well-formed answers.
- **DPO is offline preference learning:** simple and stable-looking, but it can lower the likelihood of good answers. Anchor it, or use on-policy RL.
- **Test-time compute is a second scaling axis:** pass@N with a verifier and majority vote both trade inference compute for accuracy.
- **RL sharpens:** it mostly turns "sometimes right" into "usually right" (pass@1), and the gap often shrinks at large N.

## 🎤 Explain it to a friend
Explain to a friend who has never trained a model how you taught a tiny network to add better *without showing it any new answers*. Just let it guess several times per question, check each guess with a calculator, and nudge it toward the guesses that beat its own average. Then explain why this trick does nothing when all the guesses are equally right or equally wrong. One paragraph, no equations.

## 💼 Interview corner
**Q1. How does GRPO estimate advantages, and why doesn't it need a value network? What happens for a prompt where all $G$ samples are correct?**
<details><summary>Answer</summary>

For each prompt it samples a group of $G$ completions and uses the group's own reward statistics as the baseline: $\hat A_i = (r_i - \text{mean}(r))/\text{std}(r)$. That baseline is prompt-specific and free, so no learned critic is needed (PPO would train a value model about the size of the policy). If all samples get the same reward, every advantage is zero and that prompt contributes no policy-gradient signal (only the KL term acts), which wastes compute. Hence filtering or dynamic sampling (as in DAPO) and difficulty curricula.
</details>

**Q2. When would you choose DPO over PPO/GRPO, and vice versa?**
<details><summary>Answer</summary>

DPO fits when you have a fixed dataset of preference pairs (e.g. human chat preferences) and want a simple, cheap, offline training run with no sampling loop and no reward model. GRPO/PPO fit when you can score *fresh* samples automatically (math answers, unit tests) and want on-policy exploration: the model learns from its own current mistakes. Online RL is more expensive and finicky but tends to be stronger for reasoning; DPO can suffer from likelihood displacement and distribution shift because its data comes from a different policy.
</details>

**Q3. What do the clipping and the KL penalty each do in the GRPO objective? What might happen with $\beta = 0$?**
<details><summary>Answer</summary>

Clipping the importance ratio $\rho$ to $[1-\epsilon, 1+\epsilon]$ limits how far one batch of samples can move the policy when you take several gradient steps on it, a local trust region. The KL penalty to a frozen reference is a global leash that keeps the policy close to the SFT model, which protects fluency and general ability and makes reward hacking harder. With $\beta=0$ (used in several 2025 recipes for verifiable tasks) training can move faster, but you rely entirely on the reward being un-hackable and on clipping for stability.
</details>

## 🐝 Fun fact
In the DeepSeek-R1 paper, DeepSeek-R1-Zero, trained with pure RL (GRPO) on top of a base model with **no** supervised reasoning data, raised its AIME 2024 pass@1 from **15.6% to 71.0%** over training. The authors describe an "aha moment" where the model spontaneously learned to stop, re-evaluate and correct its own reasoning mid-solution.

## 📚 Further reading
- Shao et al. (2024). *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models.* (introduces GRPO)
- DeepSeek-AI (2025). *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning.*
- Schulman et al. (2017). *Proximal Policy Optimization Algorithms.*
- Rafailov et al. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model.*
- Pang et al. (2024). *Iterative Reasoning Preference Optimization.* (DPO + NLL on chosen answers)
- Ouyang et al. (2022). *Training language models to follow instructions with human feedback.* (InstructGPT / RLHF)
- Lambert et al. (2024). *Tülu 3: Pushing Frontiers in Open Language Model Post-Training.* (RL with verifiable rewards)
- Yu et al. (2025). *DAPO: An Open-Source LLM Reinforcement Learning System at Scale.*
- Liu et al. (2025). *Understanding R1-Zero-Like Training: A Critical Perspective.* (Dr. GRPO)
- Wang et al. (2023). *Self-Consistency Improves Chain of Thought Reasoning in Language Models.*
- Chen et al. (2021). *Evaluating Large Language Models Trained on Code.* (unbiased pass@k)
- Snell et al. (2024). *Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters.*
- Hugging Face TRL documentation — `GRPOTrainer` and `DPOTrainer`.